# PuLP 4.0.0a9 補足ベンチマーク

本ノートブックは、本文で触れた **PuLP 4.0.0a9 + HiGHS** の補足検証です。  
環境差分を切り離すため、**このノートブック単体で再現できる形** にしています。

対象は、本文と同じ **500 製品・9 日ウィンドウ・7 日スライド** のローリングホライゾン生産計画です。

| 指標 | PuLP 3.3.2 | PuLP 4.0.0a9 |
|------|-----------:|-------------:|
| total wall time | 6.83 s | 12.17 s |
| build avg | 59.6 ms | 47.9 ms |
| solve avg | 70.9 ms | 188.6 ms |

ここでの関心は、**build はやや改善しているのに、solve 呼び出し全体は大きく遅くなっている理由** を確認することです。


## 0. インストール

Google Colab では次を実行してください。PuLP 4 は alpha 版なので、版を固定しています。


In [ ]:
!pip install --quiet "PuLP==4.0.0a9" "highspy==1.13.1" numpy


## 1. インポート


In [ ]:
import statistics
import time
from dataclasses import asdict, dataclass

import numpy as np
import pulp
from pulp import constants
from pulp.apis.highs import highspy

print(f"PuLP   : {pulp.__version__}")
print(f"highspy: {highspy.Highs().version()}")


## 2. ベンチマークデータ

本文と同じ乱数 seed・同じ problem size を使います。


In [ ]:
N_PRODUCTS = 500
N_DAYS = 365
WINDOW = 9
SLIDE = 7

RNG = np.random.default_rng(42)
demand = RNG.integers(10, 100, size=(N_PRODUCTS, N_DAYS)).astype(float)
max_inventory = RNG.integers(200, 500, size=N_PRODUCTS).astype(float)
initial_inventory = (max_inventory * 0.3).astype(float)
production_cost = RNG.uniform(1.0, 5.0, size=N_PRODUCTS)
inventory_cost = RNG.uniform(0.1, 0.5, size=N_PRODUCTS)

n_windows = len(range(0, N_DAYS - WINDOW + 1, SLIDE))
print(f"{N_PRODUCTS} 製品 x {N_DAYS} 日, {n_windows} windows")


## 3. ローリングホライゾン実装（PuLP 4 alpha）

`pulp.HiGHS` を使うので、ここでの solver 呼び出しは **file 経由ではなく highspy の in-memory API** です。


In [ ]:
def solve_window_pulp4(start: int, inventory_init: np.ndarray):
    end = min(start + WINDOW, N_DAYS)
    n_days = end - start
    days = list(range(n_days))
    products = list(range(N_PRODUCTS))

    t_build_start = time.perf_counter()

    prob = pulp.LpProblem("production_planning", pulp.LpMinimize)

    prod = prob.add_variable_dicts("prod", (products, days), lowBound=0)
    inv = {
        (p, t): prob.add_variable(
            f"inv_{p}_{t}",
            lowBound=0,
            upBound=float(max_inventory[p]),
        )
        for p in products
        for t in days
    }

    prob += pulp.lpSum(
        production_cost[p] * prod[p][t] + inventory_cost[p] * inv[p, t]
        for p in products
        for t in days
    )

    for p in products:
        for t in days:
            prev_inv = inventory_init[p] if t == 0 else inv[p, t - 1]
            prob += inv[p, t] == prev_inv + prod[p][t] - demand[p, start + t]

    t_build_end = time.perf_counter()

    solver = pulp.HiGHS(msg=False)
    prob.solve(solver)

    t_solve_end = time.perf_counter()

    inv_end = np.array([pulp.value(inv[p, SLIDE - 1]) for p in products])
    build_time = t_build_end - t_build_start
    solve_time = t_solve_end - t_build_end
    return pulp.value(prob.objective), inv_end, build_time, solve_time


def run_rolling_horizon_pulp4():
    inventory = initial_inventory.copy()
    objectives = []
    build_times = []
    solve_times = []

    for start in range(0, N_DAYS - WINDOW + 1, SLIDE):
        obj, inventory, bt, st = solve_window_pulp4(start, inventory)
        objectives.append(obj)
        build_times.append(bt)
        solve_times.append(st)

    return {
        "objectives": objectives,
        "build_times": build_times,
        "solve_times": solve_times,
    }


## 4. ベンチマーク実行


In [ ]:
t0 = time.perf_counter()
result = run_rolling_horizon_pulp4()
total_time = time.perf_counter() - t0

bt = np.array(result["build_times"])
st = np.array(result["solve_times"])

print(f"PuLP 4.0.0a9 (HiGHS): {len(result['objectives'])} windows")
print(f"  total wall time: {total_time:.2f} s")
print(f"  build avg      : {bt.mean()*1000:.1f} ms")
print(f"  solve avg      : {st.mean()*1000:.1f} ms")
print(f"  first build    : {bt[0]*1000:.1f} ms")
print(f"  update avg     : {bt[1:].mean()*1000:.1f} ms")


## 5. `actualSolve()` の profiling

build 済みの PuLP モデルを `pulp.HiGHS.actualSolve()` がどこで時間を使っているかを計測します。

見たいのは次の切り分けです。

- solver 作成
- PuLP の汎用表現から solver 固有形式への変換
- HiGHS 自体の実行
- 解の取り出し


In [ ]:
@dataclass
class PhaseTimes:
    create_solver: float = 0.0
    exported_variables: float = 0.0
    id_to_col: float = 0.0
    add_columns: float = 0.0
    constraints_iter: float = 0.0
    constraint_items: float = 0.0
    add_rows: float = 0.0
    call_solver: float = 0.0
    find_solution: float = 0.0
    total_actual_solve: float = 0.0


def build_problem():
    days = list(range(WINDOW))
    products = list(range(N_PRODUCTS))

    prob = pulp.LpProblem("production_planning", pulp.LpMinimize)
    prod = prob.add_variable_dicts("prod", (products, days), lowBound=0)
    inv = {
        (p, t): prob.add_variable(
            f"inv_{p}_{t}",
            lowBound=0,
            upBound=float(max_inventory[p]),
        )
        for p in products
        for t in days
    }

    prob += pulp.lpSum(
        production_cost[p] * prod[p][t] + inventory_cost[p] * inv[p, t]
        for p in products
        for t in days
    )

    for p in products:
        for t in days:
            prev_inv = initial_inventory[p] if t == 0 else inv[p, t - 1]
            prob += inv[p, t] == prev_inv + prod[p][t] - demand[p, t]

    return prob


class TimedHiGHS(pulp.HiGHS):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.phase_times = PhaseTimes()

    def createAndConfigureSolver(self, lp):
        t0 = time.perf_counter()
        out = super().createAndConfigureSolver(lp)
        self.phase_times.create_solver += time.perf_counter() - t0
        return out

    def buildSolverModel(self, lp):
        inf = highspy.kHighsInf
        obj_mult = -1 if lp.sense == constants.LpMaximize else 1

        t0 = time.perf_counter()
        exported_vars = lp.exported_variables()
        self.phase_times.exported_variables += time.perf_counter() - t0

        t0 = time.perf_counter()
        id_to_col = {v.id: j for j, v in enumerate(exported_vars)}
        self.phase_times.id_to_col += time.perf_counter() - t0

        t0 = time.perf_counter()
        for j, var in enumerate(exported_vars):
            lb = var.lowBound
            ub = var.upBound
            lp.solverModel.addCol(
                obj_mult * lp.objective.get(var, 0.0),
                -inf if lb is None else lb,
                inf if ub is None else ub,
                0,
                [],
                [],
            )
            if var.cat == constants.LpInteger and self.mip:
                lp.solverModel.changeColIntegrality(j, highspy.HighsVarType.kInteger)
        self.phase_times.add_columns += time.perf_counter() - t0

        t0 = time.perf_counter()
        constraints = list(lp.constraints())
        self.phase_times.constraints_iter += time.perf_counter() - t0

        items_elapsed = 0.0
        add_rows_elapsed = 0.0
        for constraint in constraints:
            t_items = time.perf_counter()
            non_zero_constraint_items = [
                (id_to_col[var.id], coefficient)
                for var, coefficient in constraint.items()
                if coefficient != 0
            ]
            items_elapsed += time.perf_counter() - t_items

            if len(non_zero_constraint_items) == 0:
                indices, coefficients = [], []
            else:
                indices, coefficients = zip(*non_zero_constraint_items)

            lb = constraint.getLb()
            ub = constraint.getUb()
            t_row = time.perf_counter()
            lp.solverModel.addRow(
                -inf if lb is None else lb,
                inf if ub is None else ub,
                len(indices),
                indices,
                coefficients,
            )
            add_rows_elapsed += time.perf_counter() - t_row

        self.phase_times.constraint_items += items_elapsed
        self.phase_times.add_rows += add_rows_elapsed

    def callSolver(self, lp):
        t0 = time.perf_counter()
        out = super().callSolver(lp)
        self.phase_times.call_solver += time.perf_counter() - t0
        return out

    def findSolutionValues(self, lp):
        t0 = time.perf_counter()
        out = super().findSolutionValues(lp)
        self.phase_times.find_solution += time.perf_counter() - t0
        return out

    def actualSolve(self, lp, **kwargs):
        t0 = time.perf_counter()
        out = super().actualSolve(lp, **kwargs)
        self.phase_times.total_actual_solve += time.perf_counter() - t0
        return out


def median_report(samples):
    keys = PhaseTimes().__dict__.keys()
    return {
        key: statistics.median(sample[key] for sample in samples) * 1000
        for key in keys
    }


In [ ]:
samples = []
repeats = 7

for _ in range(repeats):
    prob = build_problem()
    solver = TimedHiGHS(msg=False)
    prob.solve(solver)
    samples.append(asdict(solver.phase_times))

med = median_report(samples)

print(f"PuLP 4.0.0a9 + HiGHS profiling (median over {repeats} runs)")
print()
for key, value in med.items():
    print(f"{key:20s}: {value:8.2f} ms")

python_build_solver = (
    med["exported_variables"]
    + med["id_to_col"]
    + med["add_columns"]
    + med["constraints_iter"]
    + med["constraint_items"]
    + med["add_rows"]
)
print()
print(f"{'python buildSolverModel subtotal':20s}: {python_build_solver:8.2f} ms")


## 6. 読み取り

profiling の結果、支配的だったのは `exported_variables()` 自体ではなく、**PuLP の汎用データ構造を solver ごとの内部形式に変換する処理** でした。特に各変数を 1 個ずつ `addCol` で追加する部分の比重が大きく、ここが `actualSolve()` 全体を押し上げています。

参考として、本文で取得した中央値は次の通りです。

| phase | PuLP 3.3.2 | PuLP 4.0.0a9 |
|------|-----------:|-------------:|
| add_columns | 30.30 ms | 136.28 ms |
| constraint_items | 2.08 ms | 5.19 ms |
| add_rows | 16.05 ms | 16.07 ms |
| call_solver | 15.78 ms | 14.36 ms |
| total_actual_solve | 76.91 ms | 188.88 ms |

つまり、今回の差は **HiGHS が遅くなった** のではなく、**solver に渡す直前の Python 側変換処理が重くなった** ことにあります。


## 7. まとめ

- `pulp.HiGHS` はこの notebook では **in-memory 実行** です。
- PuLP 4 alpha では build はやや軽くなっています。
- 一方で `actualSolve()` 全体は悪化しており、主因は **per-variable の `addCol` ループ** でした。
- alpha 版なので、正式版までにここが改善される可能性はあります。

本文の主張を崩す材料ではなく、むしろ **Python 側インターフェース実装の差がベンチマークに強く出る** ことを補強する結果です。
